# Lab 10: GPT Generation

**Module 10** | Read `notes/10-gpt-causal-lms.pdf` before running this notebook.

Load a small causal language model and compare greedy decoding, temperature sampling, and nucleus (top-p) sampling.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## GPT-2 text generation

Causal language models predict the next token given all previous tokens. At inference time we **decode** autoregressively: sample or select one token, append it, and repeat. Two knobs dominate output diversity:

- **Temperature** scales logits before softmax. Low temperature (0.3) sharpens the distribution toward likely tokens; high temperature (1.2) flattens it and increases surprise.
- **Top-p (nucleus) sampling** keeps the smallest set of tokens whose cumulative probability exceeds `p` (here 0.9) and samples only within that set, trimming the long tail without a fixed top-k cutoff.

We compare three temperature settings at fixed `top_p=0.9` and collect outputs in a table.


GPT-2 small (~124M parameters) fits most lab machines. If memory is tight the cell below falls back to DistilGPT-2, which is smaller but follows the same generation API.


In [ ]:
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path("..").resolve()
PROMPT = "The future of machine learning is"
TEMPERATURES = [0.3, 0.7, 1.2]
TOP_P = 0.9
MAX_NEW_TOKENS = 60

try:
    model_name = "gpt2"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
except Exception:
    model_name = "distilgpt2"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
model.to(device)
model.eval()
print(f"Loaded {model_name}, {sum(p.numel() for p in model.parameters()):,} parameters")


For each temperature we run nucleus sampling with the same prompt and seed so differences reflect decoding settings, not random initialization. `do_sample=True` enables stochastic decoding; `top_p` applies nucleus filtering on every step.


In [ ]:
rows = []
inputs = tokenizer(PROMPT, return_tensors="pt").to(device)

for temp in TEMPERATURES:
    torch.manual_seed(42)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(42)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=temp,
            top_p=TOP_P,
            pad_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    rows.append({
        "temperature": temp,
        "top_p": TOP_P,
        "prompt": PROMPT,
        "generated_text": text,
    })
    print(f"\n--- temperature={temp}, top_p={TOP_P} ---")
    print(text)

df = pd.DataFrame(rows)
df


Lower temperature outputs should read more predictable and repetitive; higher temperature introduces varied word choices and occasional digressions. Top-p prevents sampling from extremely unlikely tokens while still allowing multiple continuations, a practical default for open-ended generation.
